In [1]:
from paths import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import (skeleton_weight_filename, naanobot_weight_filename,
                                          skeleton_config, naanobot_config, radial_config)

In [2]:
skeleton_weight_filename = skeleton_weight_filename
skeleton_cfg = skeleton_config
naanobot_weight_filename = naanobot_weight_filename
naanobot_config = naanobot_config
radial_cfg = radial_config

In [3]:
model = NanoMaker(skeleton_weight_filename=skeleton_weight_filename,
                  naanobot_weight_filename=naanobot_weight_filename,
                  skeleton_cfg=skeleton_config,
                  naanobot_config=naanobot_config,
                  radial_cfg=radial_cfg)

In [4]:
drug_i_want_to_deliver = "CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C"

In [5]:
model.ingest_chemical(drug_i_want_to_deliver)

Chemical Ingested: CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C
Drug Likeness Rules Passed: 4 / 4


In [6]:
pocket_data = model.generate_pocket_data(sampling_temp=0.3)
print(len(pocket_data))
print(type(pocket_data))

4
<class 'dict'>


In [7]:
pocket_data

{'SMILES': 'CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C',
 'Sampling_temperature': 0.3,
 '3D_skeleton': [[13.385139424333369, -7.674277957933273, 0.2760564892882574],
  [11.978347151749974, 8.376567753720797, -0.22028507483107532],
  [13.402390956874644, 3.4088477650481894, -2.914812707704371],
  [12.25191671698747, 6.153856780300708, 1.0706754859267704],
  [-4.216478892152481, -12.371681474284985, 2.798659777483089],
  [8.065391421029462, -8.748487945767929, -4.835047644034522],
  [8.472153857754355, -9.40872678570402, -2.664587991308013],
  [11.800104549154298, -3.8309052228436276, -3.8891534230172824],
  [11.361345280999199, -4.925417117479055, -2.0728149123533237],
  [10.93050458629302, -6.057697157529087, -0.10726313989604828],
  [10.290552024575227, 6.139213824830898, -3.8118466214332143],
  [2.8078220661843796, 11.712811883182376, -0.968704101398666],
  [11.032670913770568, -1.9594204850578132, 3.0099466271186572],
  [3.4655833806090057, -9.867201653174778, 5.421321577912

In [8]:
smiles_str = pocket_data["SMILES"]
skeleton = pocket_data['3D_skeleton']
aa_seq = pocket_data['aa_ids']

In [9]:
# GENERATION QUALITY ASSESSMENTS

In [12]:
import torch
import torch.nn.functional as F
from nano_maker.naanobot import NAAnoBot
from nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

@torch.no_grad()
def diagnose_generation(model, map4_enc, sph_coordinates, n=50, sampling_temperature=1.0, verbose=True):
    # explicitly force eval + disable dropout
    model.eval()
    for module in model.modules():
        if isinstance(module, torch.nn.Dropout):
            module.p = 0.0

    map4_enc = map4_enc.to(next(model.parameters()).device)

    bioch_context = [model.naano_module.get_nAAnovector("VOID") for _ in range(model.block_size)]
    coord_context = [[model.max_angstroms, 0, 0] for _ in range(model.block_size)]

    aa_chain = ""
    position_log = []  # track full run for post-hoc analysis

    for i, coordinate in enumerate(sph_coordinates[:n]):
        naano_X = model.naano_module.get_nAAno_X(
            coord_context, bioch_context, coordinate
        ).unsqueeze(0).to(map4_enc.device)

        output, _ = model.forward(naano_X, map4_enc)
        vector = output[0].detach().float()

        # distances to all amino acids
        distances = {}
        for aa_id, n_v in model.naano_module.nAAno_vectors.items():
            if aa_id == 'VOID':
                continue
            n_v_t = torch.tensor(n_v, dtype=torch.float32)
            distances[aa_id] = F.mse_loss(vector, n_v_t).item()

        sorted_distances = sorted(distances.items(), key=lambda x: x[1])
        top_aa, top_dist = sorted_distances[0]

        if verbose:
            print(f"\nPosition {i} | coord: ({coordinate[0]:.2f}Å, {coordinate[1]:.3f}rad, {coordinate[2]:.3f}rad)")
            print(f"  raw output:  {vector.numpy().round(2)}")
            print(f"  top 5:       {sorted_distances[:5]}")
            print(f"  best match:  {top_aa} (mse={top_dist:.4f})")
            print(f"  temperature: {sampling_temperature}")

        # stochastic selection at specified temperature
        aa_id = model.naano_module.approx_id(output[0], sampling_temperature=sampling_temperature)

        # flag if sampling diverged from greedy best
        if verbose and aa_id != top_aa:
            print(f"  sampled:     {aa_id} (diverged from greedy {top_aa})")

        bioch_context = bioch_context[1:] + [model.naano_module.get_nAAnovector(aa_id)]
        coord_context = coord_context[1:] + [coordinate]
        aa_chain += aa_id

        position_log.append({
            "position": i,
            "coordinate": coordinate,
            "greedy": top_aa,
            "sampled": aa_id,
            "top5": sorted_distances[:5],
            "diverged": aa_id != top_aa
        })

    if verbose:
        print(f"\nFinal chain: {aa_chain}")
        divergences = sum(1 for p in position_log if p["diverged"])
        print(f"Greedy/sample divergences: {divergences}/{len(position_log)} positions ({100*divergences/len(position_log):.1f}%)")

    return aa_chain, position_log


map4_enc = torch.tensor(smiles_fingerprint(drug_i_want_to_deliver), dtype=torch.float32).unsqueeze(0)
nb_cfg = naanobot_config.copy()
sph_coordinates = model._pocket_sph_skeleton()
naanobot_model = NAAnoBot(n_embd=nb_cfg["n_embd"], n_head=nb_cfg["n_head"],
                                           n_layers=nb_cfg["n_layers"],
                                           block_size=nb_cfg["block_size"],
                                           map4_res=nb_cfg["map4_res"],
                                           n_spatial_features=nb_cfg["n_spatial_features"],
                                           max_angstroms=nb_cfg["max_angstroms"],
                                           dropout=nb_cfg["dropout"],
                          loss_temperature=nb_cfg["loss_temperature"])
diagnose_generation(model=naanobot_model, map4_enc=map4_enc, sph_coordinates=sph_coordinates)


Position 0 | coord: (15.61Å, -0.232rad, 1.504rad)
  raw output:  [ 0.15 -0.03  1.2  -1.75 -0.13 -0.69 -0.05 -0.65  0.33  0.48 -0.61  0.86
 -0.04 -0.18  0.15 -0.4   0.27 -0.3  -0.99  0.94  0.33 -0.65 -0.26 -0.77
 -0.55 -0.14  1.14  0.47 -0.71  0.78  0.38]
  top 5:       [('P', 0.5664650797843933), ('S', 0.6329372525215149), ('T', 0.6758104562759399), ('C', 0.6835146546363831), ('Q', 0.6873471140861511)]
  best match:  P (mse=0.5665)
  temperature: 1.0
  sampled:     W (diverged from greedy P)

Position 1 | coord: (14.77Å, 0.145rad, 1.562rad)
  raw output:  [-0.02  0.16  1.1  -1.87  0.01 -0.76  0.07 -0.4   0.31  0.42 -0.43  1.08
 -0.15 -0.35  0.4  -0.19  0.13 -0.27 -1.15  1.04  0.17 -0.31 -0.57 -0.59
 -0.25 -0.14  1.33  0.77 -0.79  1.    0.19]
  top 5:       [('P', 0.5655980110168457), ('S', 0.6374998688697815), ('T', 0.6799269914627075), ('Q', 0.6868636012077332), ('N', 0.6948955655097961)]
  best match:  P (mse=0.5656)
  temperature: 1.0
  sampled:     F (diverged from greedy P)

Posi

('WFRLCATCSDVSARCNCGRPGMDNCECASQASCDQPYM',
 [{'position': 0,
   'coordinate': [15.614550590515137,
    -0.23239225149154663,
    1.5044536590576172],
   'greedy': 'P',
   'sampled': 'W',
   'top5': [('P', 0.5664650797843933),
    ('S', 0.6329372525215149),
    ('T', 0.6758104562759399),
    ('C', 0.6835146546363831),
    ('Q', 0.6873471140861511)],
   'diverged': True},
  {'position': 1,
   'coordinate': [14.772017478942871, 0.1446041762828827, 1.562303066253662],
   'greedy': 'P',
   'sampled': 'F',
   'top5': [('P', 0.5655980110168457),
    ('S', 0.6374998688697815),
    ('T', 0.6799269914627075),
    ('Q', 0.6868636012077332),
    ('N', 0.6948955655097961)],
   'diverged': True},
  {'position': 2,
   'coordinate': [14.297269821166992, 0.07643739879131317, 1.501477837562561],
   'greedy': 'P',
   'sampled': 'R',
   'top5': [('P', 0.5777061581611633),
    ('S', 0.6425110101699829),
    ('T', 0.6913866996765137),
    ('Q', 0.6977634429931641),
    ('N', 0.7029744982719421)],
   'diverg